# 10.4 · Docker in one hour (optional)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/10-deployment-mlops/04-docker/lesson.ipynb)

Read the [README](README.md) first. Docker is a separate program, so this notebook does what it can without it (inspect the files, check them, estimate the image size) and only builds and runs the image if Docker is installed and running on your machine. Every Docker command is also shown so you can run it in a terminal.

In [1]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
repo_file("data/download.py")                                  # churn_model finds the data through it
sys.path.insert(0, str(repo_file("lib/churn_model.py").parent))
import churn_model as cm

bundle = cm.load_bundle()          # trains and saves models/churn_bundle.joblib the first time
ROOT = cm.repo_root()
print(f"Model trained {bundle['trained']}, test AUC {bundle['metrics']['test_auc']}, "
      f"threshold {bundle['threshold']}")

Model trained 2026-10-28, test AUC 0.741, threshold 0.07


## 1. The three files

In [2]:
docker_dir = repo_file("modules/10-deployment-mlops/04-docker/Dockerfile").parent
dockerfile = (docker_dir / "Dockerfile").read_text()
ignore = (docker_dir / "Dockerfile.dockerignore").read_text()
requirements = (docker_dir / "requirements-api.txt").read_text()

print(dockerfile)

# ShopWave churn API in a container (lesson 10.4).
# Build from the course folder (the repo root), not from this folder:
#   docker build -f modules/10-deployment-mlops/04-docker/Dockerfile -t shopwave-churn .
#   docker run --rm -p 8000:8000 shopwave-churn
FROM python:3.12-slim

WORKDIR /app

# 1. Libraries first: this layer is cached until requirements-api.txt changes
COPY modules/10-deployment-mlops/04-docker/requirements-api.txt .
RUN pip install --no-cache-dir -r requirements-api.txt

# 2. Our code and the trained model (run train_churn.py before building)
COPY lib/churn_model.py lib/
COPY modules/10-deployment-mlops/02-fastapi/api.py .
COPY models/churn_bundle.joblib models/
ENV CHURN_MODEL_PATH=/app/models/churn_bundle.joblib

# 3. Do not run as root inside the container
RUN useradd --create-home appuser
USER appuser

EXPOSE 8000
CMD ["python", "-m", "uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"]



In [3]:
print(requirements)
print(ignore)

# Only what the API needs, pinned to the exact versions the model was saved with.
# A pickled scikit-learn model should be loaded by the same scikit-learn version (lesson 10.1).
fastapi==0.162.0
uvicorn==0.54.0
pydantic==2.14.1
scikit-learn==1.9.1
scipy==1.18.1
numpy==2.5.4
pandas==3.1.0
joblib==1.6.0

# Docker sends the whole build folder to the builder. Our build folder is the course
# root, with the data, the virtual environment and every model, so ignore everything
# and let in only the four files the Dockerfile copies.
*
!lib/churn_model.py
!modules/10-deployment-mlops/02-fastapi/api.py
!modules/10-deployment-mlops/04-docker/requirements-api.txt
!models/churn_bundle.joblib



## 2. Check before you build

Every `COPY` source must exist, or the build fails half-way. The paths are relative to the build folder: the course root. The model file must be trained first (10.1).

In [4]:
copies = [line.split()[1] for line in dockerfile.splitlines() if line.startswith("COPY")]
for source in copies:
    print(f"{'ok     ' if (ROOT / source).exists() else 'MISSING'} {source}")
allowed = [line[1:] for line in ignore.splitlines() if line.startswith("!")]
print("\nlet in by the ignore file:", sorted(allowed) == sorted(copies))

ok      modules/10-deployment-mlops/04-docker/requirements-api.txt
ok      lib/churn_model.py
ok      modules/10-deployment-mlops/02-fastapi/api.py
ok      models/churn_bundle.joblib

let in by the ignore file: True


The ignore file matters more than it looks. The build folder is the whole course, so without it Docker would send everything to the builder first:

In [5]:
def folder_mb(path):
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1e6

for name in [".venv", "data", "models"]:
    if (ROOT / name).exists():
        print(f"{name:8s} {folder_mb(ROOT / name):8.0f} MB")
print(f"sent with the ignore file: {sum((ROOT / s).stat().st_size for s in copies) / 1e3:.0f} KB")

.venv        3495 MB
data          192 MB
models        289 MB
sent with the ignore file: 13 KB


## 3. How big will the image be?

The image is the base Python image plus the libraries plus our few files. We can measure the libraries already installed here. This is a lower bound: inside a Linux container the same libraries took 397 MB, because Linux packages bundle their own copies of maths libraries (the `numpy.libs` and `scipy.libs` folders).

In [6]:
from importlib.metadata import distribution

def installed_mb(package):
    """Disk space used by an installed package and its files, in MB."""
    dist = distribution(package)
    return sum((dist.locate_file(f).stat().st_size for f in dist.files
                if dist.locate_file(f).exists()), 0) / 1e6

pins = [line.split("==")[0] for line in requirements.splitlines() if "==" in line]
sizes = pd.Series({p: installed_mb(p) for p in pins}).sort_values(ascending=False)
print(sizes.round(1).to_string())
print(f"total: {sizes.sum():.0f} MB")

scipy           97.6
pandas          77.0
scikit-learn    46.0
numpy           32.6
pydantic         3.9
joblib           1.9
fastapi          1.6
uvicorn          0.7
total: 261 MB


Our code and model are a few kilobytes. Almost all of the image is scikit-learn's maths stack (SciPy, NumPy, pandas). That is why the Dockerfile installs them in their own step: Docker caches that layer and only redoes it when `requirements-api.txt` changes.

## 4. How the container finds the model

Inside the container there is no course folder, so `churn_model.py` cannot search upwards for it. The Dockerfile sets `CHURN_MODEL_PATH` instead. Same code, different environment variable:

In [7]:
import os, importlib
os.environ["CHURN_MODEL_PATH"] = "/app/models/churn_bundle.joblib"
print("in the container:", importlib.reload(cm).MODEL_PATH)
del os.environ["CHURN_MODEL_PATH"]
print("on this machine :", importlib.reload(cm).MODEL_PATH.relative_to(ROOT))

in the container: /app/models/churn_bundle.joblib
on this machine : models/churn_bundle.joblib


## 5. Build and run (only if Docker is running)

In [8]:
import shutil, subprocess
docker = shutil.which("docker")
running = docker is not None and subprocess.run([docker, "info"], capture_output=True).returncode == 0
print("docker installed:", docker is not None, "| engine running:", running)

docker installed: True | engine running: True


In [9]:
import time, urllib.request, json
if running:
    subprocess.run([docker, "build", "-q", "-f", str(docker_dir / "Dockerfile"), "-t", "shopwave-churn", str(ROOT)], check=True)
    size = subprocess.run([docker, "image", "inspect", "shopwave-churn", "--format", "{{.Size}}"],
                          capture_output=True, text=True).stdout
    print(f"image size: {int(size) / 1e6:.0f} MB")
    cid = subprocess.run([docker, "run", "-d", "--rm", "-p", "8000:8000", "shopwave-churn"],
                         capture_output=True, text=True).stdout.strip()
    time.sleep(5)
    print(json.load(urllib.request.urlopen("http://127.0.0.1:8000/health")))
    subprocess.run([docker, "stop", cid], capture_output=True)
else:
    print("Docker is not running here, so nothing was built. Run the commands below in a terminal.")

sha256:f26f5a139000bc4d4236b290e9c976cf187a70ac666e66e32dd20810773e7526
image size: 155 MB


{'status': 'ok', 'model_trained': '2026-10-28', 'test_auc': 0.741, 'threshold': 0.07, 'features': ['orders', 'spend', 'products', 'tenure_days', 'days_since_last']}


## 6. The commands

From the course root (the folder with `README.md`), on Windows, macOS or Linux:

```bash
python modules/10-deployment-mlops/01-saving-and-reproducibility/train_churn.py     # the model must exist first
docker build -f modules/10-deployment-mlops/04-docker/Dockerfile -t shopwave-churn .
docker images shopwave-churn                     # see its size
docker run --rm -p 8000:8000 shopwave-churn      # Ctrl+C to stop
```

Then, in another terminal: `curl http://127.0.0.1:8000/health` (macOS/Linux) or `Invoke-RestMethod http://127.0.0.1:8000/health` (Windows PowerShell), or open http://127.0.0.1:8000/docs in a browser.

## Quiz answers

1. The image is the packaged recipe result (files, libraries, settings); a container is one running instance of it. You can run many containers from one image.
2. So that Docker can reuse the cached library layer when only the code or model changes; installing libraries is the slow step.
3. The build folder is the whole course, with the virtual environment and data: hundreds of MB to send for four small files. The ignore file sends only what the Dockerfile copies.
4. The model was pickled by scikit-learn 1.9.1; loading it with another version may fail or behave differently. Pinning makes the container match the training environment.
5. If someone breaks into the web service, they get an ordinary user's permissions inside the container, not root's.

**Next:** [10.5 Experiment tracking with MLflow](../05-mlflow/)